# Challenge 6 - ReadyNow! Emergency Preparedness Assistant

Architecture: User -> root coordinator -> weather, search, and route specialists -> critique/refine workflow. Callbacks validate input and log interactions.

In [ ]:
%pip install -q --upgrade google-adk google-cloud-aiplatform[agent_engines,adk]
import os, requests
from datetime import datetime, timezone
import vertexai
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools import google_search, agent_tool
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.genai import types
from vertexai.preview import reasoning_engines
PROJECT_ID=os.environ.get('GOOGLE_CLOUD_PROJECT','qwiklabs-gcp-02-9e12deb8c42f'); LOCATION='us-central1'; MODEL='gemini-2.5-flash'
vertexai.init(project=PROJECT_ID,location=LOCATION); AUDIT_LOG=[]
def message_text(llm_request):
    for content in reversed(llm_request.contents or []):
        if content.role=='user': return ' '.join(p.text or '' for p in content.parts or [])
    return ''
def validate_and_log(callback_context:CallbackContext,llm_request:LlmRequest):
    text=message_text(llm_request); AUDIT_LOG.append({'time':datetime.now(timezone.utc).isoformat(),'input':text})
    if any(x in text.lower() for x in ('ignore previous','system prompt','jailbreak','api key')): return LlmResponse(content=types.Content(role='model',parts=[types.Part(text='I only assist with emergency preparedness and safety information.')]))
    return None
def log_response(callback_context,llm_response): AUDIT_LOG.append({'time':datetime.now(timezone.utc).isoformat(),'stage':'response'}); return None


In [ ]:
weather_agent=LlmAgent(name='weather',model=MODEL,instruction='Give current U.S. weather safety guidance.')
search_agent=LlmAgent(name='search',model=MODEL,instruction='Use Google Search for current official emergency information.',tools=[google_search])
route_agent=LlmAgent(name='routes',model=MODEL,instruction='Give preparedness route guidance and require users to follow official evacuation orders.')
critique=LlmAgent(name='critique',model=MODEL,instruction='Check {research_draft} for safety and accuracy.',output_key='critique_notes')
refine=LlmAgent(name='refine',model=MODEL,instruction='Refine {research_draft} using {critique_notes}.',output_key='final_answer')
workflow=SequentialAgent(name='refinement',sub_agents=[critique,refine])
root_agent=LlmAgent(name='readynow_root',model=MODEL,instruction='Coordinate emergency preparedness help. Delegate to specialists and use the refinement workflow.',sub_agents=[weather_agent,route_agent,workflow],tools=[agent_tool.AgentTool(agent=search_agent)],before_model_callback=validate_and_log,after_model_callback=log_response)
app=reasoning_engines.AdkApp(agent=root_agent); print('ReadyNow created.')


In [ ]:
s=app.create_session(user_id='challenge-six-tester'); sid=s['id'] if isinstance(s,dict) else s.id
for event in app.stream_query(user_id='challenge-six-tester',session_id=sid,message='Give hurricane preparedness advice for a family in Miami, Florida.'):
    print(event)
print('Audit log entries:',len(AUDIT_LOG))